# Homework 1 — Download the books

Download the CSV file containing the book links and then download all PDFs into the `books/` directory.

In [2]:
from pathlib import Path
from urllib.request import Request, urlopen

CSV_URL = (
    "https://raw.githubusercontent.com/alexeygrigorev/"
    "ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"
)

DATA_DIR = Path("books")
CSV_PATH = Path("books.csv")
DATA_DIR.mkdir(exist_ok=True)

request = Request(CSV_URL, headers={"User-Agent": "Mozilla/5.0"})
with urlopen(request) as response:
    CSV_PATH.write_bytes(response.read())

print(f"Downloaded: {CSV_PATH}")

Downloaded: books.csv


In [3]:
import csv
import re

with CSV_PATH.open(newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

print(f"Books found: {len(rows)}")
print(rows[0])

Books found: 7
{'title': 'Think Python 2e', 'book_url': 'https://greenteapress.com/wp/think-python-2e/', 'pdf_url': 'https://greenteapress.com/thinkpython2/thinkpython2.pdf'}


In [4]:
def safe_filename(value):
    value = re.sub(r"[^\w\s-]", "", value.lower())
    return re.sub(r"[-\s]+", "-", value).strip("-")

for row in rows:
    title = row.get("title") or row.get("name") or "book"
    url = row.get("pdf_url")
    output_path = DATA_DIR / f"{safe_filename(title)}.pdf"

    if output_path.exists():
        print(f"Skipping existing file: {output_path}")
        continue

    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})

    try:
        with urlopen(request, timeout=60) as response:
            output_path.write_bytes(response.read())
        print(f"Downloaded: {output_path}")
    except Exception as error:
        print(f"Failed to download {title}: {error}")

Downloaded: books\think-python-2e.pdf
Downloaded: books\think-dsp.pdf
Downloaded: books\think-complexity-2e.pdf
Downloaded: books\think-java-2e.pdf
Downloaded: books\physical-modeling-in-matlab.pdf
Downloaded: books\think-os.pdf
Downloaded: books\think-c.pdf


In [5]:
from markitdown import MarkItDown
from pathlib import Path

TEXT_DIR = Path("books_text")
TEXT_DIR.mkdir(exist_ok=True)
md = MarkItDown()

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    md_path = TEXT_DIR / pdf_path.with_suffix(".md").name
    result = md.convert(pdf_path)   
    md_path.write_text(
        result.text_content,
        encoding="utf-8"
    )
    print(f"Converted: {md_path}")   

Converted: books_text\physical-modeling-in-matlab.md
Converted: books_text\think-c.md
Converted: books_text\think-complexity-2e.md
Converted: books_text\think-dsp.md
Converted: books_text\think-java-2e.md
Converted: books_text\think-os.md
Converted: books_text\think-python-2e.md


In [6]:
books = []

for md_path in sorted(TEXT_DIR.glob("*.md")):
    lines = md_path.read_text(encoding="utf-8").splitlines()
    content = [line for line in lines if line.strip()]

    books.append({
        "source": md_path.name,
        "content": content,
    })

print(f"Books processed: {len(books)}")
print(books[0])

Books processed: 7
{'source': 'physical-modeling-in-matlab.md', 'content': ['| Physical | Modeling | in MATLAB |', '| -------- | -------- | --------- |', 'Version 4.0', 'Allen B. Downey', 'Green Tea Press', 'Needham, Massachusetts', 'Physical Modeling in MATLAB', 'Copyright 2012, 2019, 2021 Allen B. Downey', 'Green Tea Press', '9 Washburn Ave', 'Needham MA 02492', 'Permissionisgrantedtocopy, distribute, and/ormodifythisdocumentunderthetermsofthe', 'Creative Commons Attribution-NonCommercial 4.0 Unported License, which is available at', 'https://greenteapress.com/matlab/license.', 'This book was typeset by the author using pdflatex, among other free, open-source programs.', 'The LaTeX source for this book is available from https://greenteapress.com/matlab.', 'Theinformationinthisbookisdistributedonan“AsIs” basis, withoutwarranty. Whileevery', 'precaution has been taken in the preparation of this work, neither the author nor No Starch', 'Press, Inc. shall have any liability to any person

In [7]:
from gitsource import chunk_documents
chunks = chunk_documents(books, size=100,step=50)
chunks

[{'start': 0,
  'content': ['| Physical | Modeling | in MATLAB |',
   '| -------- | -------- | --------- |',
   'Version 4.0',
   'Allen B. Downey',
   'Green Tea Press',
   'Needham, Massachusetts',
   'Physical Modeling in MATLAB',
   'Copyright 2012, 2019, 2021 Allen B. Downey',
   'Green Tea Press',
   '9 Washburn Ave',
   'Needham MA 02492',
   'Permissionisgrantedtocopy, distribute, and/ormodifythisdocumentunderthetermsofthe',
   'Creative Commons Attribution-NonCommercial 4.0 Unported License, which is available at',
   'https://greenteapress.com/matlab/license.',
   'This book was typeset by the author using pdflatex, among other free, open-source programs.',
   'The LaTeX source for this book is available from https://greenteapress.com/matlab.',
   'Theinformationinthisbookisdistributedonan“AsIs” basis, withoutwarranty. Whileevery',
   'precaution has been taken in the preparation of this work, neither the author nor No Starch',
   'Press, Inc. shall have any liability to any 

In [8]:
from collections import Counter

chunks_by_source = Counter(
    chunk.get("source", "unknown")
    for chunk in chunks
)

for source, number_of_chunks in sorted(chunks_by_source.items()):
    print(f"{source}: {number_of_chunks} chunks")

physical-modeling-in-matlab.md: 119 chunks
think-c.md: 107 chunks
think-complexity-2e.md: 144 chunks
think-dsp.md: 111 chunks
think-java-2e.md: 250 chunks
think-os.md: 68 chunks
think-python-2e.md: 189 chunks


In [9]:
from minsearch import Index


def prepare_documents(chunks):
    documents = []

    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)

    return documents


documents = prepare_documents(chunks)


index = Index(text_fields=["content"])
index.fit(documents)
len(documents)

988

In [10]:
results = index.search("python function definition", num_results=5)
results

[{'start': 1800,
  'content': "| ------ | ------------ | --- | --- |\n| total  | = total +    | a;  |     |\nend\nans = total\nIfyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised\nto find, after running the script, that their values had changed. If you have two scripts that\n48 Functions\nuse the same variable names, you might find that they work separately and then break when\nyou try to combine them. This kind of interaction is called a collision.\nname\nAs the number of scripts you write increases, and they get longer and more complex, name\ncollisions become more of a problem. Avoiding this problem is one of several motivations for\nfunctions.\n| 5.2 | Defining | Functions |     |     |\n| --- | -------- | --------- | --- | --- |\nA function is like a script, except that each function has its own workspace, so any variables\ndefined inside a function only exist while the function is running and don’t interfere with\nvariables in other workspaces, even 

In [11]:
from openai import OpenAI

openai_client = OpenAI()

In [12]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )
    print(response.usage.input_tokens)
    return {
        "text" : response.output_text,
        "input_tokens" : response.usage.input_tokens,
        "output_tokens" : response.usage.output_tokens
    }

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [13]:
result = rag('python function definition')
result

7928


{'text': "To define a function in Python, you use the `def` keyword followed by the function name and parentheses containing any parameters. Here's a basic structure:\n\n```python\ndef function_name(parameters):\n    # Code block of the function\n    return output\n```\n\n### Example\nHere's a simple example of a function that adds two numbers:\n\n```python\ndef add_numbers(a, b):\n    result = a + b\n    return result\n```\n\n### Key Points\n- **Function Definition**: Begins with `def`, followed by the function name and parameters.\n- **Code Block**: Indented lines following the function definition that perform the tasks.\n- **Return Statement**: Used to send a result back to the caller.\n\nFunctions created this way have their own local variable scope, so variable names used within the function won't interfere with variables outside.",
 'input_tokens': 7928,
 'output_tokens': 172}

In [14]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [15]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions,outputtype, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=outputtype
    )
    
    return {
        "data" : response.output_parsed,
        "input_tokens" : response.usage.input_tokens,
        "output_tokens" : response.usage.output_tokens
    }

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions, RAGResponse)
    return answer

In [16]:
result = rag('python function definition')
result

{'data': RAGResponse(answer='### Python Function Definition\nTo define a function in Python, you follow these basic steps:\n\n1. **Use the `def` keyword:** This indicates to Python that you\'re defining a function.\n2. **Specify the function name:** This should be a descriptive name that helps indicate what the function does.\n3. **Add parentheses with parameters:** If your function needs inputs (arguments), you include them inside parentheses.\n4. **Add a colon:** To signify the start of the function body.\n5. **Write the function body:** Indented under the function definition, this contains the code that is executed when the function is called.\n6. **Optionally return a value:** Use the `return` keyword to send a result back to the caller.\n\n### Example\nHere\'s a simple example of a function that adds two numbers:\n```python\ndef add(x, y):\n    """Return the sum of x and y."""\n    return x + y\n```\n### Usage\nYou can call the function by using its name and passing the required a